# Random Forest

Train and tune a Random Forest classifier for flight cancellation prediction. We run on both feature sets (with and without `IS_COVID`) and compare.

The workflow for each version:
1. Train a default RF as a baseline
2. Run a randomised hyperparameter search on the full training set
3. Retrain the best config on all training data
4. Tune the decision threshold on validation (optimise F2)
5. Report final metrics on the test set

**Heads up:** running `RandomizedSearchCV` on 2M+ rows with 100 iterations and 3-fold CV means ~300 full RF fits.

In [ ]:
import os
import numpy as np
import pandas as pd
import joblib

from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import average_precision_score, fbeta_score
from sklearn.model_selection import RandomizedSearchCV, StratifiedKFold

MODEL_DIR = 'model_weights'
os.makedirs(MODEL_DIR, exist_ok=True)

## Load the preprocessed data

Both feature versions come from the single `artifacts/` directory. Targets are shared.

In [ ]:
datasets = {}
for version in ['with_covid', 'no_covid']:
    datasets[version] = {
        'X_train': pd.read_parquet(f'artifacts/X_train_{version}.parquet'),
        'X_val':   pd.read_parquet(f'artifacts/X_val_{version}.parquet'),
        'X_test':  pd.read_parquet(f'artifacts/X_test_{version}.parquet'),
    }

y_train = pd.read_parquet('artifacts/y_train.parquet')['CANCELLED'].values
y_val   = pd.read_parquet('artifacts/y_val.parquet')['CANCELLED'].values
y_test  = pd.read_parquet('artifacts/y_test.parquet')['CANCELLED'].values

print(f'Train: {len(y_train):,} rows  |  cancel rate: {y_train.mean():.2%}')
print(f'Val:   {len(y_val):,} rows  |  cancel rate: {y_val.mean():.2%}')
print(f'Test:  {len(y_test):,} rows  |  cancel rate: {y_test.mean():.2%}')
print(f'\nWith COVID: {datasets["with_covid"]["X_train"].shape[1]} features')
print(f'No COVID:   {datasets["no_covid"]["X_train"].shape[1]} features')

## Approach

For each feature version we:
- Start with a vanilla RF (`class_weight='balanced'`) to set a baseline
- Search over a broad hyperparameter grid using `RandomizedSearchCV` with `scoring='average_precision'` (PR-AUC)
- Retrain the best params on the full training set
- Sweep thresholds on validation to find the one that maximises F2 (since missing a cancellation is worse than a false alarm)
- Evaluate on the held-out test set

In [ ]:
def find_best_threshold(y_true, y_proba, beta=2):
    """Sweep thresholds in [0, 1] and return the one that maximises F-beta."""
    best_t, best_score = 0.5, 0.0
    for t in np.linspace(0, 1, 101):
        score = fbeta_score(y_true, (y_proba >= t).astype(int), beta=beta, zero_division=0)
        if score > best_score:
            best_score, best_t = score, t
    return best_t, best_score


def evaluate_on_test(model, X_test, y_test, threshold):
    """Get PR-AUC and F2 on the test set at a given threshold."""
    proba = model.predict_proba(X_test)[:, 1]
    preds = (proba >= threshold).astype(int)
    return {
        'PR-AUC': average_precision_score(y_test, proba),
        'F2': fbeta_score(y_test, preds, beta=2, zero_division=0),
        'Threshold': threshold,
    }

## Train and tune

We loop over both feature versions. The hyperparameter grid is intentionally broad — we let `RandomizedSearchCV` sample 100 combinations and pick the best by cross-validated PR-AUC.\n\nThe search runs on a 400K stratified sample (not the full 2M+) to keep runtime reasonable. Hyperparameter rankings are stable enough at this size, and we retrain the winner on the full dataset anyway.

In [ ]:
param_grid = {
    'n_estimators': [400, 600, 800, 1000],
    'max_depth': [15, 25, 35, None],
    'max_features': ['sqrt', 'log2', 0.3, 0.5],
    'min_samples_split': [2, 5, 10, 20],
    'min_samples_leaf': [1, 2, 4, 8],
    'class_weight': [
        {0: 1, 1: 15},
        {0: 1, 1: 20},
        {0: 1, 1: 30},
        {0: 1, 1: 40},
        'balanced_subsample',
    ],
}

cv = StratifiedKFold(n_splits=3, shuffle=True, random_state=42)
all_results = []

for version in ['with_covid', 'no_covid']:
    X_tr = datasets[version]['X_train']
    X_v  = datasets[version]['X_val']
    X_te = datasets[version]['X_test']

    print(f'\n{"=" * 60}')
    print(f'  {version}  ({X_tr.shape[1]} features)')
    print(f'{"=" * 60}')

    # --- Baseline: default RF with balanced class weights ---
    print('\n[1] Baseline RF...')
    rf_base = RandomForestClassifier(
        class_weight='balanced', n_jobs=-1, random_state=42)
    rf_base.fit(X_tr, y_train)

    val_proba = rf_base.predict_proba(X_v)[:, 1]
    base_t, base_val_f2 = find_best_threshold(y_val, val_proba)
    base_test = evaluate_on_test(rf_base, X_te, y_test, base_t)
    print(f'    Val F2: {base_val_f2:.4f} (threshold={base_t:.2f})')
    print(f'    Test PR-AUC: {base_test["PR-AUC"]:.4f}  |  Test F2: {base_test["F2"]:.4f}')

    all_results.append({
        'Version': version, 'Model': 'Baseline RF', **base_test})

    # --- Hyperparameter search on full training data ---
    # use a 400K sample for the search to keep runtime manageable
    sample_size = min(400_000, len(X_tr))
    sample_idx = X_tr.sample(sample_size, random_state=42).index
    X_sample = X_tr.loc[sample_idx]
    y_sample = y_train[sample_idx]

    print(f'\n[2] RandomizedSearchCV (50 iterations, 3-fold CV on {len(X_sample):,} sample)...')

    search = RandomizedSearchCV(
        RandomForestClassifier(random_state=42, n_jobs=-1),
        param_grid,
        n_iter=50,
        scoring='average_precision',
        cv=cv,
        n_jobs=-1,
        random_state=42,
        verbose=1,
    )
    search.fit(X_sample, y_sample)

    print(f'    Best CV PR-AUC: {search.best_score_:.4f}')
    print(f'    Best params: {search.best_params_}')

    # --- Retrain best config on full training data ---
    print('\n[3] Retraining best config on full training data...')
    best_rf = RandomForestClassifier(
        **search.best_params_, random_state=42, n_jobs=-1)
    best_rf.fit(X_tr, y_train)

    val_proba = best_rf.predict_proba(X_v)[:, 1]
    tuned_t, tuned_val_f2 = find_best_threshold(y_val, val_proba)
    tuned_test = evaluate_on_test(best_rf, X_te, y_test, tuned_t)
    print(f'    Val F2: {tuned_val_f2:.4f} (threshold={tuned_t:.2f})')
    print(f'    Test PR-AUC: {tuned_test["PR-AUC"]:.4f}  |  Test F2: {tuned_test["F2"]:.4f}')

    all_results.append({
        'Version': version, 'Model': 'Tuned RF', **tuned_test})

    # --- Save the tuned model ---
    out_path = f'{MODEL_DIR}/random_forest_{version}.pkl'
    joblib.dump(best_rf, out_path)
    print(f'\n    Saved: {out_path}')

## Results

How did the baseline compare to the tuned model? And did the COVID flag make any difference?

In [ ]:
results_df = pd.DataFrame(all_results)
print(results_df.to_string(index=False))